# 🏏 HawkVision: Cricket Ball Trajectory Prediction on Google Colab (GPU)

This notebook runs the core computer vision & trajectory prediction engine of **HawkVision** using Google Colab's free GPU acceleration (NVIDIA T4 / V100 / A100).

### ⚡ Step 0: Ensure GPU is Enabled
Before running the cells below, go to the top menu:
1. Click **Runtime** → **Change runtime type**
2. Under **Hardware accelerator**, select **T4 GPU**
3. Click **Save**

In [ ]:
# Step 1: Verify GPU Availability
import torch
print("=" * 50)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU   : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU for 10x faster inference!")
print("=" * 50)

In [ ]:
# Step 2: Install Required Libraries
# Google Colab comes with PyTorch, OpenCV, and NumPy pre-installed.
# We only need Ultralytics (YOLOv8) and imageio-ffmpeg for browser-compatible H.264 video encoding.
!pip install ultralytics imageio-ffmpeg

### 📁 Step 3: Get Project Files

You have two options to load your project files in Colab:

**Option A (Recommended): Clone your Git repository**
```bash
!git clone <YOUR_GITHUB_REPO_URL>
%cd <REPO_NAME>
```

**Option B: Upload the zip archive directly**
If you uploaded a `.zip` of the project to Colab, run the cell below to unzip it:

In [ ]:
# If using a zip archive, uncomment and run:
# !unzip -q hawkeye.zip -d /content/hawkvision
# %cd /content/hawkvision

# Verify project files are present
import os
required_files = ["predict.py", "trajectory_tracker.py", "speed_tracker.py"]
for f in required_files:
    status = "✅ Found" if os.path.exists(f) else "❌ Missing"
    print(f"{f:<25}: {status}")

In [ ]:
# Step 4: Run Core Prediction on GPU
# This runs the complete pipeline: YOLOv8 ball detection, delivery window trimming,
# 2-phase Kalman trajectory prediction, pitch bounce reflection, and speed kinematics.

!python predict.py \
    --video videos/test1.mp4 \
    --model runs/detect/train5/weights/best.pt \
    --output videos/output_predicted.mp4 \
    --no-show

In [ ]:
# Step 5: Watch the Annotated Hawkeye Video Directly in Colab
from IPython.display import HTML
from base64 import b64encode

output_video_path = "videos/output_predicted.mp4"

if os.path.exists(output_video_path):
    mp4 = open(output_video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <div style="text-align: center; background: #0f172a; padding: 15px; border-radius: 12px;">
        <h3 style="color: #f59e0b; margin-top: 0;">🏏 HawkVision Trajectory Analysis</h3>
        <video width="640" controls autoplay muted loop style="border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.5);">
            <source src="{data_url}" type="video/mp4">
        </video>
    </div>
    """))
else:
    print(f"File not found: {output_video_path}")

In [ ]:
# Step 6 (Optional): Download Processed Video to Your Local Machine
from google.colab import files
files.download("videos/output_predicted.mp4")

### 🎯 Upload & Test Your Own Cricket Videos
Run the cell below to upload any custom video file from your computer and run GPU analysis on it:

In [ ]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    out_name = f"output_{filename}"
    print(f"\n🚀 Running GPU Trajectory Analysis on: {filename}...")
    !python predict.py \
        --video "{filename}" \
        --model runs/detect/train5/weights/best.pt \
        --output "{out_name}" \
        --no-show
    
    if os.path.exists(out_name):
        mp4 = open(out_name, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        display(HTML(f"""
        <div style="text-align: center; background: #0f172a; padding: 15px; border-radius: 12px; margin-top: 15px;">
            <h3 style="color: #38bdf8; margin-top: 0;">🏏 Output: {out_name}</h3>
            <video width="640" controls autoplay muted loop style="border-radius: 8px;">
                <source src="{data_url}" type="video/mp4">
            </video>
        </div>
        """))
